# 04 · The tensile-test workflow as a knowledge graph

**Goal.** Show what [semantikon](https://github.com/pyiron/semantikon) records about a workflow recipe and about a run, how unit and quantity-kind annotations become RDF, what semantikon's validation catches, and how a run graph is queried together with the data graph it was computed from.

**Audience.** Ontologists and RDF experts, and workflow developers.

**Prerequisites.**
- RDF and SPARQL basics.
- The structure of the workflow (atomic nodes, the `specimen_workflow` macro, the batch) as introduced in notebook 03. Ontologists can read this notebook without it; the first section summarizes what is needed.
- The `pmd-demonstrator` environment from `environment.yml`, which installs the package `pmd_tensile`.
- `ONTODOCKER_ADDRESS` (and `ONTODOCKER_TOKEN`, if the instance needs one) set in the environment before Jupyter starts. Neither is printed here.

**Reading paths.** Ontologists: [01](01-tensile-data-from-graphs.ipynb) → **04** → [05](05-tensile-analysis-overview.ipynb). Workflow developers: [03](03-tensile-workflow-building.ipynb) → **04** → [02](02-tensile-analysis-nodes.ipynb) → [05](05-tensile-analysis-overview.ipynb).

*Execution class: network-read.* The notebook queries the PMDco 2 tensile-test datasets on Ontodocker, fetches one of them as Turtle, and downloads curve CSV files from the public data host. Graph files are written only to a temporary directory that the notebook creates and removes; nothing is written to remote services or to the repository.

## Setup

In [ ]:
import os
import re
import tempfile
import warnings
from pathlib import Path

import flowrep as fr
import pandas as pd
import pyiron_workflow as pwf
import rdflib
from IPython.display import display
from semantikon import u

from pmd_tensile import analysis, demo

# semantikon's provenance hashing triggers an upstream networkx notice about changed hashes.
warnings.filterwarnings("ignore", message="The hashes produced for directed graphs", category=UserWarning)
pd.set_option("display.precision", 4)
pd.set_option("display.max_colwidth", None)  # pandas would otherwise cut IRIs, URLs and hashes.

if not os.environ.get(demo.ONTODOCKER_ADDRESS_ENV, "").strip():
    raise RuntimeError("Set ONTODOCKER_ADDRESS (and ONTODOCKER_TOKEN if required) before starting Jupyter.")

PROFILE_ID = "pmdco2-tto"
SOURCE = {
    "kind": "ontodocker",
    "datasets": [
        "pmdco2_tto_example_parallel",
        "pmdco2_tto_example_perpendicular",
        "pmdco2_tto_example_diagonal",
    ],
}
SPECIMEN_IDS = ["Zx3", "Zd3"]  # the reference case and the specimen traced in section 5

PREFIXES = {
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "owl": "http://www.w3.org/2002/07/owl#",
    "sns": "http://pyiron.org/ontology/",
    "pmdco": "https://w3id.org/pmd/co/PMD_",
    "bfo": "http://purl.obolibrary.org/obo/BFO_",
    "ro": "http://purl.obolibrary.org/obo/RO_",
    "iao": "http://purl.obolibrary.org/obo/IAO_",
    "obi": "http://purl.obolibrary.org/obo/OBI_",
    "qudt": "http://qudt.org/schema/qudt/",
    "unit": "http://qudt.org/vocab/unit/",
    "qk": "http://qudt.org/vocab/quantitykind/",
    "xsd": "http://www.w3.org/2001/XMLSchema#",
}
workdir = tempfile.TemporaryDirectory(prefix="tensile-kg-")  # removed in the last cell
print("Ontology profile:", PROFILE_ID)
print("Datasets:", ", ".join(SOURCE["datasets"]))
print("Specimens:", ", ".join(SPECIMEN_IDS))

## 1. A recipe graph and a run graph

The tensile analysis runs in two stages (notebook 03): an *exploration* that checks the source and lists the specimens, and a *batch* with one `specimen_workflow` per specimen (24 steps: acquisition, analysis, comparison with the lab values). Here the batch covers two specimens.

semantikon turns each stage into RDF in two forms:

- the **recipe** graph: what will run, i.e. the steps, their ports and how they are connected, with the Python function behind each step;
- the **run** graph: what ran, i.e. additionally the input and output values of every step and a provenance hash for each of them.

semantikon uses its own workflow vocabulary: PMDco 3 terms for functions, ports and data, OBO terms (BFO, RO, IAO, OBI) for the relations, and QUDT for units and quantity kinds. It is independent of the ontology of the tensile-test data, which here is PMDco 2. Credentials are not workflow inputs, so they cannot enter either graph: section 4 lists the inputs of the run, and `tests/test_separation.py` checks with a dummy token that it appears in no graph, output or table.

In [ ]:
retrieved_at = pd.Timestamp.now(tz="Europe/Berlin").floor("s")
tensile = demo.run_tensile_analysis(PROFILE_ID, SOURCE, specimen_ids=SPECIMEN_IDS)
print("Data retrieved:", retrieved_at)

graphs = {
    "exploration recipe": demo.recipe_knowledge_graph(tensile.exploration_node),
    "exploration run": demo.run_knowledge_graph(tensile.exploration_run),
    "batch recipe": demo.recipe_knowledge_graph(tensile.batch_workflow),
    "batch run": demo.run_knowledge_graph(tensile.batch_run),
}
for graph in graphs.values():
    for prefix, namespace in PREFIXES.items():
        graph.bind(prefix, namespace, override=True)

HAS_VALUE = rdflib.URIRef(PREFIXES["pmdco"] + "0000006")
HASH_CLASS = rdflib.URIRef(PREFIXES["iao"] + "0020000")
sizes = pd.DataFrame(
    {
        name: {
            "triples": len(graph),
            "values (pmdco:0000006)": len(set(graph.subject_objects(HAS_VALUE))),
            "hashes (iao:0020000)": len(set(graph.subjects(rdflib.RDF.type, HASH_CLASS))),
        }
        for name, graph in graphs.items()
    }
).T
display(sizes)

In the recipe graph, the values (`pmdco:0000006`) are function and module names, docstrings, a constant and hashes of the functions. The run graph adds the value of every input and output and a hash for each. A characteristic example is the output `e_gpa` (E, the regression slope of the elastic part) of the step `final_elastic_fit` of Zd3. The triples below are those whose subject is that output port, its data node, or the data node's hash and quantity-kind individuals. Both graphs contain the same port and data-node *classes* (IRIs starting with `sns:W…`); the individuals differ (`sns:<hash>_…`), and only the run's data node has a value and a hash:

In [ ]:
# Batch nodes are named specimen_<i> in the order of SPECIMEN_IDS.
E_OUTPUT = f"specimen_{SPECIMEN_IDS.index('Zd3')}-final_elastic_fit_0-outputs-e_gpa"


def short(term, graph: rdflib.Graph) -> str:
    # Prefixed form of a term; the long middle part of IRIs and labels of E_OUTPUT is shown as "…".
    if isinstance(term, rdflib.Literal):
        return term.n3(graph.namespace_manager)
    text = str(term)
    for prefix, namespace in PREFIXES.items():
        if text.startswith(namespace):
            text = f"{prefix}:{text[len(namespace):]}"
            break
    if E_OUTPUT in text:
        head, tail = text.split(E_OUTPUT, 1)
        text = head[:10] + "…e_gpa" + tail  # e.g. sns:9a528b…e_gpa_data
    return text


def triples_about(graph: rdflib.Graph, fragment: str) -> pd.DataFrame:
    # Triples of the individuals whose IRI contains `fragment` (classes, sns:W..., and labels left out).
    rows = [
        {"subject": short(s, graph), "predicate": short(p, graph), "object": short(o, graph)}
        for s, p, o in graph
        if isinstance(s, rdflib.URIRef) and fragment in str(s) and p != rdflib.RDFS.label
        and not str(s).startswith(PREFIXES["sns"] + "W")
    ]
    return pd.DataFrame(rows).sort_values(["subject", "predicate"], ignore_index=True)


for name in ("batch recipe", "batch run"):
    print(name)
    display(triples_about(graphs[name], E_OUTPUT))

## 2. Units and quantity kinds as RDF

The analysis nodes declare units and QUDT quantity kinds in their type hints, e.g. `e_gpa` as `u(float, units="GPa", uri=QK_MODULUS)` with `QK_MODULUS` the quantity kind `ModulusOfElasticity` (notebook 03, section 1). The quantity kind classifies the dimension; it does not make the regression slope E a material modulus. semantikon writes them in two places:

- on the **data node** (an individual): `qudt:hasUnit unit:GigaPA`, and `obi:0001927` to an individual typed `qk:ModulusOfElasticity` (visible in the tables above);
- on the **data-node class**: OWL restrictions on the same two properties. Validation (section 3) turns these restrictions into SHACL shapes.

The restrictions of the class of Zd3's `e_gpa` data node in the recipe graph:

In [ ]:
recipe = graphs["batch recipe"]
data_class = next(
    s for s in set(recipe.subjects(rdflib.RDFS.subClassOf))
    if str(s).startswith(PREFIXES["sns"] + "W") and str(s).endswith(E_OUTPUT + "_data")
)
print(short(data_class, recipe))
for parent in sorted(recipe.objects(data_class, rdflib.RDFS.subClassOf), key=lambda term: isinstance(term, rdflib.BNode)):
    if isinstance(parent, rdflib.BNode):
        restriction = sorted(
            f"{short(p, recipe)} {short(o, recipe)}" for p, o in recipe.predicate_objects(parent) if p != rdflib.RDF.type
        )
        print("    rdfs:subClassOf [ a owl:Restriction ;", " ; ".join(restriction), "]")
    else:
        print("    rdfs:subClassOf", short(parent, recipe))

Because units and quantity kinds are triples, they can be queried without knowing the analysis code. Example query (b) of the package lists them for every output of the analysis functions (module `pmd_tensile.analysis`) in the run graph:

In [ ]:
units = demo.run_example_query(graphs["batch run"], "analysis_output_units")
units = units.replace(r"^http://qudt\.org/vocab/(unit|quantitykind)/", "", regex=True)
display(units)

For example, which analysis outputs are stresses in MPa? A filter on the query result answers it; the same condition can be written into the SPARQL query:

In [ ]:
display(units[(units["quantity_kind"] == "Stress") & (units["unit"] == "MegaPA")][["function", "output"]])

## 3. Validation

`demo.validate_knowledge_graph` calls `semantikon.validate_values`: it applies RDFS reasoning and checks the graph against SHACL shapes that semantikon derives from the OWL restrictions of section 2 and from shapes contained in the graph. It works on a copy, so the graph itself is unchanged. The four graphs of section 1 conform:

In [ ]:
def violations(graph: rdflib.Graph) -> list[str]:
    # One line per SHACL violation: the focus node (without its hash prefix) and the message.
    conforms, report = demo.validate_knowledge_graph(graph)
    if conforms:
        return []
    focus, lines = None, []
    for line in report.splitlines():
        line = line.strip()
        if line.startswith("Focus Node:"):
            focus = line.removeprefix("Focus Node:").strip().split("_", 1)[-1]
        elif line.startswith("Message:"):
            message = re.sub(r"^Node \S+->", "", line.removeprefix("Message:").strip())
            lines.append(f"{focus}: {message}")
    return lines


def verdict(found: list[str]) -> str:
    return "conforms" if not found else f"{len(found)} violation{'s' if len(found) > 1 else ''}"


for name, graph in graphs.items():
    before = len(graph)
    found = violations(graph)
    print(f"{name}: {verdict(found)} (triples before and after: {before}, {len(graph)})")

Does validation catch a connection that makes no physical sense? Three wrong connections test it. Each wires the output of one node into an input that declares a unit and a quantity kind:

- **stress into force:** the package node `analysis.engineering_stress` twice, the stress output (MPa, `Stress`) of the first into the force input (N, `Force`) of the second;
- **kN into N:** a node that converts a force to kN, into a node that expects N; the quantity kind is the same, only the unit differs;
- **no unit into N:** a node without annotations, into a node that expects N.

A fourth workflow, `newton_into_newton`, is the correct connection, for comparison. The three helper nodes (`to_kilonewton`, `unannotated_copy`, `stress_from_force`) are defined here.

In [ ]:
@fr.atomic
def to_kilonewton(force_n: u(float, units="N", uri=analysis.QK_FORCE)) -> u(float, units="kN", uri=analysis.QK_FORCE):
    force_kn = force_n / 1000.0
    return force_kn


@fr.atomic
def unannotated_copy(value: float) -> float:
    copied = value * 1.0
    return copied


@fr.atomic
def stress_from_force(force_n: u(float, units="N", uri=analysis.QK_FORCE), s0_mm2: analysis.AreaMM2) -> analysis.StressMPa:
    stress_mpa = force_n / s0_mm2
    return stress_mpa


@fr.workflow
def stress_into_force(force_n, s0_mm2):
    stress_mpa = analysis.engineering_stress(force_n, s0_mm2)
    wrong = analysis.engineering_stress(stress_mpa, s0_mm2)
    return wrong


@fr.workflow
def kilonewton_into_newton(force_n, s0_mm2):
    force_kn = to_kilonewton(force_n)
    stress_mpa = stress_from_force(force_kn, s0_mm2)
    return stress_mpa


@fr.workflow
def no_unit_into_newton(force_n, s0_mm2):
    copied = unannotated_copy(force_n)
    stress_mpa = stress_from_force(copied, s0_mm2)
    return stress_mpa


@fr.workflow
def newton_into_newton(force_n, s0_mm2):
    stress_mpa = stress_from_force(force_n, s0_mm2)
    return stress_mpa


for workflow in (stress_into_force, kilonewton_into_newton, no_unit_into_newton, newton_into_newton):
    found = violations(demo.recipe_knowledge_graph(workflow))
    print(f"{workflow.__name__} (recipe): {verdict(found)}")
    for line in found:
        print("   ", line)

The check needs only the recipe, so it can run before the workflow does. pyiron_workflow itself does not check units: it runs the kN-into-N workflow without complaint and returns a stress 1000 times too small. The run graph of that run fails validation in the same way:

In [ ]:
for workflow in (kilonewton_into_newton, newton_into_newton):
    run = pwf.node(workflow).run(force_n=2000.0, s0_mm2=10.0)
    found = violations(demo.run_knowledge_graph(run))
    print(f"{workflow.__name__}: stress {run.outputs.stress_mpa} MPa; run graph: {verdict(found)}")
    for line in found:
        print("   ", line)

What this shows about the check:

- It compares, for every connection, the unit and the quantity kind that the receiving input declares with those carried by the data that arrives. A different quantity kind, a different unit of the same quantity kind, and an unannotated output into an annotated input were all reported. (The last follows from `validate_values`' default `strict_typing=True`, which, according to its docstring, requires the preceding output to carry the declared unit.)
- It does not convert units and does not stop a run: it is a separate step on the graph, before or after running.
- It checks the declared units and quantity kinds, not the numbers.
- Only ports with unit annotations carry restrictions. In the tensile workflow, many ports pass dictionaries, dataclasses or strings without annotations; connections between those are not checked. The share in the batch recipe:

In [ ]:
HAS_UNIT = rdflib.URIRef(PREFIXES["qudt"] + "hasUnit")
data_classes = {
    s for s in recipe.subjects(rdflib.RDFS.subClassOf)
    if str(s).startswith(PREFIXES["sns"] + "W") and str(s).endswith("_data")
}
with_unit = {
    c for c in data_classes
    if any((parent, rdflib.OWL.onProperty, HAS_UNIT) in recipe for parent in recipe.objects(c, rdflib.RDFS.subClassOf))
}
print(f"batch recipe: {len(with_unit)} of {len(data_classes)} data-node classes carry a unit restriction")

## 4. Example queries

The package has four documented SPARQL queries over the workflow graphs (`demo.EXAMPLE_QUERIES`). They use only what semantikon records: each step is linked to its Python function (`ro:0000059`), the function to its name and module (`iao:0000235`), ports are parts of steps (`bfo:0000051`), and data nodes hang off ports (`ro:0000057`). Query (b) was shown in section 2.

**(a) Step roles.** Which steps are ontology-aware and which are ontology-agnostic? The role follows from the module that the graph records for each step's function:

In [ ]:
roles = pd.concat(
    [demo.run_example_query(graphs[name], "step_roles") for name in ("exploration run", "batch run")]
)
for role, functions in roles.groupby("role")["function"]:
    print(f"{role}: {', '.join(sorted(set(functions)))}")

**(c) Run inputs.** Which inputs did the run use? The inputs of the top-level batch workflow, among them the source and the complete set of method parameters:

In [ ]:
for _, row in demo.run_example_query(graphs["batch run"], "run_inputs").iterrows():
    print(f"{row['input']}: {row['value']}")

**(d) Analysis parameters.** Which parameters did each specimen's analysis receive? The query returns the value of the `analysis_parameters` step per specimen. To show that a changed parameter is recorded, Zd3 runs once more as a single-specimen workflow with a larger plastic offset (`offset_plastic_strain` = 0.003 instead of 0.002, see notebook 03):

In [ ]:
retrieved_at = pd.Timestamp.now(tz="Europe/Berlin").floor("s")
zd3_offset = demo.build_specimen_node("zd3").run(
    profile_id=PROFILE_ID, source=SOURCE, specimen_id="Zd3", parameters={"offset_plastic_strain": 0.003}
)
print("Data retrieved:", retrieved_at)

recorded = pd.concat(
    [
        demo.run_example_query(graphs["batch run"], "analysis_parameters").assign(run="batch, defaults"),
        demo.run_example_query(demo.run_knowledge_graph(zd3_offset), "analysis_parameters").assign(run="Zd3, offset 0.003"),
    ],
    ignore_index=True,
)
recorded["offset_plastic_strain"] = recorded["parameters"].str.extract(r"offset_plastic_strain=([0-9.]+)")
display(recorded[["run", "specimen_id", "offset_plastic_strain"]])
print("recorded value, abridged:", recorded.loc[0, "parameters"][:160], "...")

## 5. One query over the run graph and the data graph

The run graph records, per specimen, the URL and SHA-256 of the curve file that the analysis read, as plain strings. The PMDco 2 data graph links each test to the same URL through its CSVW curve table (`csvw:url`). That shared value lets one SPARQL query trace a computed result back to the test in the data graph and to the lab-reported value.

The data graph of the diagonal specimens is fetched from Ontodocker as Turtle and merged with the batch run graph in rdflib:

In [ ]:
retrieved_at = pd.Timestamp.now(tz="Europe/Berlin").floor("s")
client = demo.ontodocker_client()  # praeco's OntodockerClient; the package reads address and token
data_graph = rdflib.Graph().parse(data=client.datasets.fetch_turtle("pmdco2_tto_example_diagonal"), format="turtle")
print("Data retrieved:", retrieved_at)

merged = rdflib.Graph()
merged += graphs["batch run"]
merged += data_graph
print(f"run graph {len(graphs['batch run'])} + data graph {len(data_graph)} = merged {len(merged)} triples")

The query has two halves. In the run graph it finds, within one per-specimen workflow, the value and unit of `e_gpa` from `final_elastic_fit`, the curve URL and SHA-256 from `curve_channels`, and the `specimen_id` input. In the data graph it finds the tensile test whose curve table has the same URL, its test-piece name and its reported modulus of elasticity. The join is the `FILTER` on the URL; the specimen IDs of both halves must agree as well.

In [ ]:
JOINT_QUERY = '''
PREFIX bfo:   <http://purl.obolibrary.org/obo/BFO_>
PREFIX iao:   <http://purl.obolibrary.org/obo/IAO_>
PREFIX ro:    <http://purl.obolibrary.org/obo/RO_>
PREFIX pmdco: <https://w3id.org/pmd/co/PMD_>
PREFIX qudt:  <http://qudt.org/schema/qudt/>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>
PREFIX pmd:   <https://w3id.org/pmd/co/>
PREFIX csvw:  <http://www.w3.org/ns/csvw#>

SELECT ?specimen_id ?computed_e ?computed_unit ?curve_url ?curve_sha256 ?test ?lab_e ?lab_unit WHERE {
    # Run graph (semantikon): one per-specimen workflow and three of its steps' ports.
    ?workflow bfo:0000051 ?fit_step, ?channels_step, ?id_port .
    ?id_port a/rdfs:subClassOf pmdco:0000066 ; pmdco:0000128 "specimen_id" ;
             ro:0000057 [ pmdco:0000006 ?run_specimen_id ] .
    ?fit_step ro:0000059/iao:0000235 [ a pmdco:0000100 ; pmdco:0000006 "final_elastic_fit" ] ;
              bfo:0000051 ?e_port .
    ?e_port a/rdfs:subClassOf pmdco:0000067 ; pmdco:0000128 "e_gpa" ;
            ro:0000057 [ pmdco:0000006 ?computed_e ; qudt:hasUnit ?computed_unit ] .
    ?channels_step ro:0000059/iao:0000235 [ a pmdco:0000100 ; pmdco:0000006 "curve_channels" ] ;
                   bfo:0000051 ?url_port, ?sha_port .
    ?url_port a/rdfs:subClassOf pmdco:0000067 ; pmdco:0000128 "curve_url" ;
              ro:0000057 [ pmdco:0000006 ?curve_url ] .
    ?sha_port a/rdfs:subClassOf pmdco:0000067 ; pmdco:0000128 "curve_sha256" ;
              ro:0000057 [ pmdco:0000006 ?curve_sha256 ] .

    # Data graph (PMDco 2): the test whose curve table has the same URL.
    ?test a pmd:TensileTest ;
          pmd:characteristic ?dataset, ?modulus ;
          pmd:input/pmd:characteristic ?name .
    ?dataset pmd:resource/csvw:url ?table_url .
    FILTER(STR(?table_url) = STR(?curve_url))
    ?name a pmd:TestPieceName ; pmd:value ?specimen_id .
    FILTER(STR(?specimen_id) = STR(?run_specimen_id))
    ?modulus a pmd:ModulusOfElasticity ; pmd:value ?lab_e ; pmd:unit ?lab_unit .
}
ORDER BY ?specimen_id
'''

result = merged.query(JOINT_QUERY)
traced = pd.DataFrame([[str(value) for value in row] for row in result], columns=[str(v) for v in result.vars])
display(traced.T)

The query returns one row: Zd3 is the only specimen of the batch whose curve is linked in the diagonal data graph (Zx3's test is in the parallel dataset, which was not merged). The checksum comes from the run graph; the data graph records no checksum, so the SHA-256 identifies the bytes the analysis actually read.

The two graphs write the unit differently: semantikon uses the QUDT IRI `unit:GigaPA`, the PMDco 2 graph `unit:GigaPa`, which is not a QUDT IRI. A comparison of the unit IRIs would therefore fail; the `pmdco2-tto` profile maps the second to the first when it reads the data (notebook 01). With the values in the same unit, the deviation follows directly:

In [ ]:
row = traced.iloc[0]
computed_e, lab_e = float(row["computed_e"]), float(row["lab_e"])
print(f"{row['specimen_id']}: computed E {computed_e:.2f} GPa, lab-reported E {lab_e:.1f} GPa, "
      f"deviation {100 * (computed_e - lab_e) / lab_e:+.1f} %")

Zd3's computed E, a regression slope rather than a modulus, lies far below the lab value. Its elastic part is not straight, and how the lab obtained 195 GPa is unknown (notebooks 02 and 05). The query does not explain the deviation, but it shows in one step which test, which curve file and which reported value the number belongs to.

## 6. Saving the graphs: N-Triples and Turtle

Two practical points when a graph is saved:

- **Blank-node labels.** semantikon 1.4.4 names some blank nodes after IRIs. rdflib writes such labels unchanged into N-Triples, where they are not valid, so the file cannot be parsed again. `demo.write_graph` relabels these blank nodes first (blank-node labels are local to a file, so the graph is the same); reporting this upstream is an open follow-up.
- **Precision.** rdflib 7.6.0 writes `xsd:double` values in Turtle with 7 significant digits, so doubles do not survive a Turtle round trip. N-Triples keeps every literal as it is. `demo.write_graph` therefore writes N-Triples by default; Turtle is for reading.

Both points, on the batch run graph, in the temporary directory:

In [ ]:
run_graph = graphs["batch run"]
folder = Path(workdir.name)

plain = folder / "batch_run_plain.nt"
run_graph.serialize(destination=plain, format="nt", encoding="utf-8")
try:
    rdflib.Graph().parse(plain, format="nt")
    print("plain N-Triples: read back")
except Exception as error:
    print(f"plain N-Triples: cannot be read back ({type(error).__name__})")



def literals(graph: rdflib.Graph) -> set:
    return {o for _, _, o in graph if isinstance(o, rdflib.Literal)}


saved = demo.read_graph(demo.write_graph(run_graph, folder / "batch_run.nt"))
print(f"demo.write_graph: read back, {len(saved)} of {len(run_graph)} triples, "
      f"literals identical: {literals(saved) == literals(run_graph)}")

turtle = demo.parse_turtle(demo.to_turtle(run_graph))
changed = literals(run_graph) - literals(turtle)
print(f"Turtle round trip: {len(turtle)} of {len(run_graph)} triples, {len(changed)} literals changed, "
      f"datatypes: {sorted({short(literal.datatype, run_graph) for literal in changed})}")

e_node = next(s for s in run_graph.subjects(HAS_VALUE) if str(s).endswith(E_OUTPUT + "_data"))
print(f"Zd3's E: {saved.value(e_node, HAS_VALUE)} after N-Triples, {turtle.value(e_node, HAS_VALUE)} after Turtle")

## Summary

- semantikon exports a recipe graph (steps, ports, connections, functions) and a run graph (additionally every value and its provenance hash) in its own PMDco 3/OBO/QUDT vocabulary (section 1).
- Unit and quantity-kind annotations become `qudt:hasUnit` and quantity-kind individuals on the data nodes, and OWL restrictions on their classes; they can be queried, e.g. for all stress outputs in MPa (section 2).
- Validation of the recipe reported a stress wired into a force input, a kN output wired into an N input, and an unannotated output wired into an N input; pyiron_workflow ran the kN-into-N workflow and returned a wrong stress (section 3).
- The run graph records the method parameters each specimen used (section 4).
- One SPARQL query over the run graph and the hosted data graph traced Zd3's computed E to its test, its curve file with SHA-256, and the lab-reported E (section 5).

[Notebook 05](05-tensile-analysis-overview.ipynb) runs the full analysis of all ten specimens on PMDco 2 and PMDco 3; [notebook 02](02-tensile-analysis-nodes.ipynb) explains the analysis nodes.

In [ ]:
workdir.cleanup()